# Feature Distributions

**Topic:** Exploratory Data Analysis

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns

import plotly.graph_objects as go
import plotly.express as px

import ipywidgets as widgets
from ipywidgets import Dropdown, ToggleButton, Output, HBox, VBox, HTML

from IPython.display import display, clear_output
from scipy import stats

titanic = sns.load_dataset("titanic")
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this session you will be able to:

- **Assess** whether a numeric feature follows a normal distribution using a Q-Q plot and the Shapiro-Wilk test
- **Apply** a log transformation to a right-skewed feature and explain why it helps
- **Interpret** skewness and kurtosis values in the context of modeling readiness

> **Tip:** Select `fare` first and note the extreme right skew. Then toggle the log transform on. Watch both the histogram and the Q-Q plot change. A Q-Q plot where the points hug the diagonal line means the distribution is close to normal.

---
## How we got here

In `10_multivariate_analysis.ipynb` we examined relationships between multiple variables at once. Before we hand our features to a model, we need to assess whether their individual distributions are ready. This notebook draws directly on `statistics/12_normal_distribution.ipynb` and `statistics/06_shape_of_distributions.ipynb`. Those notebooks built the conceptual foundation; this one shows the practical consequences of distribution shape for modeling.

---
## Why this matters for data science

Many machine learning algorithms assume that numeric input features are roughly normally distributed. Linear regression, logistic regression, and support vector machines all perform better when features are symmetric and on comparable scales. A feature with a strong right skew (like `fare`) will cause a linear model to over-weight the rare extreme values. A log transformation compresses the tail, making the distribution more symmetric and the model more stable.

---
## Try it yourself

In [ ]:
out = Output()

numeric_cols = titanic.select_dtypes(include="number").columns.tolist()

col_dropdown = Dropdown(
    options=numeric_cols,
    value="fare",
    description="Column:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="300px"),
)

log_toggle = ToggleButton(
    value=False,
    description="Apply log transform",
    button_style="info",
    layout=widgets.Layout(width="200px"),
)

def render(col, use_log):
    raw = titanic[col].dropna()
    if use_log:
        series = np.log1p(raw)
        label = f"log(1 + {col})"
    else:
        series = raw
        label = col

    skew_val = series.skew()
    stat, p_value = stats.shapiro(series.sample(min(len(series), 200), random_state=42))

    if p_value > 0.05:
        normality_text = f"Shapiro-Wilk p = {p_value:.4f} — cannot reject normality (distribution may be normal)"
        norm_color = PALETTE["accent"]
    else:
        normality_text = f"Shapiro-Wilk p = {p_value:.4f} — rejects normality at p < 0.05 (distribution is not normal)"
        norm_color = PALETTE["secondary"]

    (osm, osr), (slope, intercept, r) = stats.probplot(series)

    fig = go.Figure()

    # Histogram
    kde_x = np.linspace(series.min(), series.max(), 400)
    kde_y = stats.gaussian_kde(series)(kde_x)
    kde_scale = len(series) * (series.max() - series.min()) / 25

    fig.add_trace(go.Histogram(
        x=series, nbinsx=25,
        marker_color=PALETTE["primary"], opacity=0.65,
        name="Distribution", xaxis="x", yaxis="y",
    ))
    fig.add_trace(go.Scatter(
        x=kde_x, y=kde_y * kde_scale,
        mode="lines", line=dict(color=PALETTE["secondary"], width=2.5),
        name="KDE", xaxis="x", yaxis="y",
    ))

    # Q-Q plot on second x/y axes
    qq_x_range = [min(osm), max(osm)]
    qq_line_y = [slope * v + intercept for v in qq_x_range]

    fig.add_trace(go.Scatter(
        x=osm, y=osr,
        mode="markers",
        marker=dict(color=PALETTE["primary"], size=5, opacity=0.5),
        name="Q-Q points", xaxis="x2", yaxis="y2",
    ))
    fig.add_trace(go.Scatter(
        x=qq_x_range, y=qq_line_y,
        mode="lines", line=dict(color=PALETTE["secondary"], width=2, dash="dash"),
        name="Normal reference line", xaxis="x2", yaxis="y2",
    ))

    layout = base_layout(
        title=f"{label}  |  skewness = {skew_val:.3f}",
        xaxis_title=label,
        yaxis_title="Count",
    )
    layout.update(
        xaxis2=dict(
            title="Theoretical Quantiles",
            title_font=dict(size=FONT["size_axis"]),
            tickfont=dict(size=FONT["size_tick"]),
            domain=[0.55, 1.0],
            gridcolor="#DEE2E6",
        ),
        yaxis2=dict(
            title="Sample Quantiles",
            title_font=dict(size=FONT["size_axis"]),
            tickfont=dict(size=FONT["size_tick"]),
            anchor="x2",
            gridcolor="#DEE2E6",
        ),
        xaxis=dict(domain=[0.0, 0.45]),
        showlegend=False,
        height=400,
    )
    fig.update_layout(layout)

    with out:
        clear_output(wait=True)
        fig.show()
        display(HTML(
            f'<div style="font-family:Inter,Arial,sans-serif;font-size:14px;'
            f'color:{norm_color};padding:8px 12px;border-left:4px solid {norm_color};'
            f'background:#F8F9FA;margin-top:8px;">{normality_text}</div>'
        ))

def on_change(change):
    render(col_dropdown.value, log_toggle.value)

col_dropdown.observe(on_change, names="value")
log_toggle.observe(on_change, names="value")

display(VBox([HBox([col_dropdown, log_toggle]), out]))
render(col_dropdown.value, log_toggle.value)

---
## What's happening?

Two tools for assessing normality:

**Q-Q plot (Quantile-Quantile plot):** Plots the sample quantiles of your data against the theoretical quantiles of a normal distribution. If the points fall along the diagonal reference line, the data is roughly normal. Curves or S-shapes indicate skewness or heavy tails. This is the tool from `statistics/12_normal_distribution.ipynb`.

**Shapiro-Wilk test:** A formal statistical test where:
- Null hypothesis: the data comes from a normal distribution
- p-value > 0.05: we cannot reject normality
- p-value < 0.05: the distribution is significantly non-normal

The log transformation:

```python
# Natural log, offset by 1 to handle zero values
series_log = np.log1p(series)  # equivalent to log(1 + x)
```

This is useful specifically for right-skewed data with a lower bound near zero, which is exactly what `fare` and most count variables look like.

---
## Real-world example: Titanic Dataset

The two panels below compare the raw fare distribution (left) with its log-transformed version (right).

Notice:
- The **raw fare distribution** is strongly right-skewed, with most passengers paying under £50 and a handful paying over £200
- After **log transformation**, the distribution becomes roughly bell-shaped and much more symmetric
- This transformation would be required before using `fare` as a feature in a linear model

> **Discussion question:** Why might a model perform better if its input features are normally distributed? Think about what "distance from the mean" means in a linear model when the distribution has a long tail.

In [ ]:
np.random.seed(42)

fare_raw = titanic["fare"].dropna()
fare_log = np.log1p(fare_raw)

kde_x_raw = np.linspace(fare_raw.min(), fare_raw.max(), 400)
kde_y_raw = stats.gaussian_kde(fare_raw)(kde_x_raw)
scale_raw = len(fare_raw) * (fare_raw.max() - fare_raw.min()) / 30

kde_x_log = np.linspace(fare_log.min(), fare_log.max(), 400)
kde_y_log = stats.gaussian_kde(fare_log)(kde_x_log)
scale_log = len(fare_log) * (fare_log.max() - fare_log.min()) / 30

fig = go.Figure()

fig.add_trace(go.Histogram(
    x=fare_raw, nbinsx=30,
    marker_color=PALETTE["secondary"], opacity=0.6,
    name="Raw fare", xaxis="x", yaxis="y",
))
fig.add_trace(go.Scatter(
    x=kde_x_raw, y=kde_y_raw * scale_raw,
    mode="lines", line=dict(color=PALETTE["secondary"], width=2),
    name="Raw KDE", xaxis="x", yaxis="y",
))

fig.add_trace(go.Histogram(
    x=fare_log, nbinsx=30,
    marker_color=PALETTE["primary"], opacity=0.6,
    name="log(1 + fare)", xaxis="x2", yaxis="y2",
))
fig.add_trace(go.Scatter(
    x=kde_x_log, y=kde_y_log * scale_log,
    mode="lines", line=dict(color=PALETTE["primary"], width=2),
    name="Log KDE", xaxis="x2", yaxis="y2",
))

fig.update_layout(
    title=dict(text="Titanic Fare: Raw vs Log-Transformed Distribution",
               font=dict(size=FONT["size_title"], family=FONT["family"])),
    xaxis=dict(title="Fare (£)", domain=[0.0, 0.45],
               gridcolor="#DEE2E6", tickfont=dict(size=FONT["size_tick"])),
    yaxis=dict(title="Count", gridcolor="#DEE2E6", tickfont=dict(size=FONT["size_tick"])),
    xaxis2=dict(title="log(1 + Fare)", domain=[0.55, 1.0], anchor="y2",
                gridcolor="#DEE2E6", tickfont=dict(size=FONT["size_tick"])),
    yaxis2=dict(title="Count", anchor="x2",
                gridcolor="#DEE2E6", tickfont=dict(size=FONT["size_tick"])),
    paper_bgcolor=PALETTE["background"],
    plot_bgcolor=PALETTE["surface"],
    showlegend=True,
    legend=dict(font=dict(size=FONT["size_legend"])),
    height=400,
    margin=dict(l=60, r=30, t=60, b=60),
)
fig.show()

### Distribution shapes and the transformations that help

| Distribution shape | Example column | Recommended transformation |
|--------------------|---------------|---------------------------|
| Right-skewed (long right tail) | fare, income, house price | log(x) or log(1+x) |
| Left-skewed (long left tail) | test scores near ceiling | square or cube |
| Bimodal (two peaks) | age in a mixed dataset | Separate the groups or use categorical bins |
| Heavy tails (high kurtosis) | extreme stock returns | Winsorize (cap at percentile) |
| Roughly symmetric | age, temperature | Usually no transformation needed |

---
## Key takeaway

> **Checking feature distributions before modeling is not optional. A skewed feature in a linear model silently distorts every coefficient that depends on it.**

---
*Next up: 12_eda_workflow_end_to_end.ipynb — a complete, fully worked EDA of the Titanic dataset from raw data to modeling insights*